# Embryo Stage Classification

This notebook trains four pretrained CNN backbones for embryo stage classification:
- MobileNetV2
- VGG16
- VGG19
- InceptionV3

It compares a weighted CrossEntropy baseline against a custom distance-aware loss designed for ordered stage labels.


In [1]:
import os
import copy
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    mobilenet_v2,
    vgg16,
    vgg19,
    inception_v3,
    MobileNet_V2_Weights,
    VGG16_Weights,
    VGG19_Weights,
    Inception_V3_Weights,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cuda


In [2]:
DATA_DIR = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/'
ANN_DIR = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations'

PHASES = [
    'tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+',
    'tM', 'tSB', 'tB', 'tEB', 'tHB'
]

label_map = {phase: idx for idx, phase in enumerate(PHASES)}

@dataclass
class Config:
    sample_rate: int = 8
    epochs: int = 4
    batch_size: int = 32
    inception_batch_size: int = 16
    image_size: int = 224
    inception_size: int = 299
    lr: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    random_state: int = 42
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    dist_lambda: float = 0.25
    aux_weight: float = 0.4
    merge_source: int = 15
    merge_target: int = 14

cfg = Config()
print(cfg)

Config(sample_rate=8, epochs=4, batch_size=32, inception_batch_size=16, image_size=224, inception_size=299, lr=0.0001, weight_decay=0.0001, num_workers=2, random_state=42, label_smoothing=0.1, grad_clip=1.0, dist_lambda=0.25, aux_weight=0.4, merge_source=15, merge_target=14)


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.random_state)

## Build Sampled Dataframe

Frames are sampled once every `sample_rate` frames inside each annotated interval.


In [4]:
def build_dataframe(sample_rate: int = 8) -> pd.DataFrame:
    rows = []

    annotation_files = sorted([f for f in os.listdir(ANN_DIR) if f.endswith('.csv')])

    for file_name in tqdm(annotation_files, desc='Reading annotations'):
        embryo_id = file_name.replace('_phases.csv', '')
        ann_path = os.path.join(ANN_DIR, file_name)
        image_dir = os.path.join(DATA_DIR, embryo_id)

        if not os.path.isdir(image_dir):
            continue

        image_files = sorted(os.listdir(image_dir))
        if not image_files:
            continue

        ann_df = pd.read_csv(ann_path, header=None, names=['phase', 'start', 'end'])

        for _, row in ann_df.iterrows():
            phase = row['phase']
            if phase not in label_map:
                continue

            start = int(row['start'])
            end = int(row['end'])
            label = label_map[phase]

            for frame_idx in range(start, end, sample_rate):
                if 0 <= frame_idx < len(image_files):
                    rows.append({
                        'image': os.path.join(image_dir, image_files[frame_idx]),
                        'label': label,
                        'embryo': embryo_id,
                    })

    return pd.DataFrame(rows)

df = build_dataframe(cfg.sample_rate)
print('Total samples:', len(df))
display(df.head())

Reading annotations:   0%|          | 0/704 [00:00<?, ?it/s]

Total samples: 40337


,image,label,embryo
0,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
1,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
2,/kaggle/input/datasets/abhishekbuddiga06/embry...,0,AA83-7
3,/kaggle/input/datasets/abhishekbuddiga06/embry...,1,AA83-7
4,/kaggle/input/datasets/abhishekbuddiga06/embry...,1,AA83-7


In [5]:
embryos = df['embryo'].unique()

train_ids, temp_ids = train_test_split(embryos, test_size=0.30, random_state=cfg.random_state, shuffle=True)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=cfg.random_state, shuffle=True)

train_df = df[df['embryo'].isin(train_ids)].copy()
val_df = df[df['embryo'].isin(val_ids)].copy()
test_df = df[df['embryo'].isin(test_ids)].copy()

print('Split sizes before merge')
print('Train:', len(train_df))
print('Val  :', len(val_df))
print('Test :', len(test_df))

Split sizes before merge
Train: 28123
Val  : 6156
Test : 6058


In [6]:
print('Train label counts before merge:')
display(train_df['label'].value_counts().sort_index())

for split_df in [train_df, val_df, test_df]:
    split_df.loc[split_df['label'] == cfg.merge_source, 'label'] = cfg.merge_target

num_classes = len(PHASES) - 1

print('Train label counts after merge:')
display(train_df['label'].value_counts().sort_index())
print('Effective num_classes:', num_classes)

Train label counts before merge:


label
0      911
1     3907
2      770
3     2716
4      598
5     2714
6      862
7      898
8     1016
9     2990
10    4671
11    1620
12    1579
13     990
14    1877
15       4
Name: count, dtype: int64

Train label counts after merge:


label
0      911
1     3907
2      770
3     2716
4      598
5     2714
6      862
7      898
8     1016
9     2990
10    4671
11    1620
12    1579
13     990
14    1881
Name: count, dtype: int64

Effective num_classes: 15


In [7]:
classes = np.arange(num_classes)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label'].values)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

weight_table = pd.DataFrame({
    'class_id': classes,
    'weight': weights,
})
display(weight_table)

,class_id,weight
0,0,2.058031
1,1,0.479874
2,2,2.434892
3,3,0.690304
4,4,3.135229
5,5,0.690813
6,6,2.175019
7,7,2.087825
8,8,1.845341
9,9,0.627046


## Dataset And Dataloaders


In [8]:
class EmbryoDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        for _ in range(5):
            row = self.frame.iloc[idx]
            try:
                image = Image.open(row['image']).convert('RGB')
                if self.transform is not None:
                    image = self.transform(image)
                label = torch.tensor(int(row['label']), dtype=torch.long)
                return image, label
            except Exception:
                idx = random.randint(0, len(self.frame) - 1)

        fallback = torch.zeros(3, cfg.image_size, cfg.image_size)
        return fallback, torch.tensor(0, dtype=torch.long)

train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

inception_train_transform = transforms.Compose([
    transforms.Resize((cfg.inception_size, cfg.inception_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

inception_eval_transform = transforms.Compose([
    transforms.Resize((cfg.inception_size, cfg.inception_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def make_loader(frame, transform, batch_size, shuffle):
    return DataLoader(
        EmbryoDataset(frame, transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

standard_loaders = {
    'train': make_loader(train_df, train_transform, cfg.batch_size, True),
    'val': make_loader(val_df, eval_transform, cfg.batch_size, False),
    'test': make_loader(test_df, eval_transform, cfg.batch_size, False),
}

inception_loaders = {
    'train': make_loader(train_df, inception_train_transform, cfg.inception_batch_size, True),
    'val': make_loader(val_df, inception_eval_transform, cfg.inception_batch_size, False),
    'test': make_loader(test_df, inception_eval_transform, cfg.inception_batch_size, False),
}

print('Standard train batches :', len(standard_loaders['train']))
print('Inception train batches:', len(inception_loaders['train']))

Standard train batches : 879
Inception train batches: 1758


## Models And Loss Functions


In [9]:
def get_model(name: str, num_classes: int):
    name = name.lower()

    if name == 'mobilenet':
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        model.classifier = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(model.last_channel, num_classes),
        )

    elif name == 'vgg16':
        model = vgg16(weights=VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    elif name == 'vgg19':
        model = vgg19(weights=VGG19_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    elif name == 'inception':
        model = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)

    else:
        raise ValueError(f'Unknown model: {name}')

    return model.to(DEVICE)

class DistanceAwareCrossEntropy(nn.Module):
    def __init__(self, class_weights, num_classes, lambda_dist=0.25, label_smoothing=0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        self.lambda_dist = lambda_dist
        self.scale = max((num_classes - 1) ** 2, 1)
        self.register_buffer('class_ids', torch.arange(num_classes, dtype=torch.float32).view(1, -1))

    def forward(self, logits, targets):
        ce_term = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        sq_dist = (self.class_ids - targets.unsqueeze(1).float()) ** 2
        dist_term = (probs * sq_dist).sum(dim=1).mean() / self.scale
        return ce_term + self.lambda_dist * dist_term

def make_loss(loss_name: str):
    if loss_name == 'ce':
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing).to(DEVICE)
    if loss_name == 'distance_ce':
        return DistanceAwareCrossEntropy(
            class_weights=class_weights,
            num_classes=num_classes,
            lambda_dist=cfg.dist_lambda,
            label_smoothing=cfg.label_smoothing,
        ).to(DEVICE)
    raise ValueError(f'Unknown loss: {loss_name}')

In [10]:
def unpack_outputs(outputs):
    if hasattr(outputs, 'logits'):
        return outputs.logits, outputs.aux_logits
    if isinstance(outputs, tuple):
        if len(outputs) == 2:
            return outputs[0], outputs[1]
        return outputs[0], None
    return outputs, None

def train_one_epoch(model, loader, optimizer, loss_fn, model_name: str):
    model.train()
    running_loss = 0.0
    preds_all, targets_all = [], []

    progress = tqdm(loader, desc=f'{model_name} train', leave=False)
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        main_logits, aux_logits = unpack_outputs(outputs)

        loss = loss_fn(main_logits, labels)
        if model_name == 'inception' and aux_logits is not None:
            loss = loss + cfg.aux_weight * loss_fn(aux_logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(main_logits, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        targets_all.extend(labels.detach().cpu().numpy())
        progress.set_postfix(loss=f'{loss.item():.4f}')

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(targets_all, preds_all)
    epoch_f1 = f1_score(targets_all, preds_all, average='weighted')
    return epoch_loss, epoch_acc, epoch_f1

@torch.no_grad()
def evaluate(model, loader, model_name: str):
    model.eval()
    preds_all, targets_all = [], []

    for images, labels in tqdm(loader, desc=f'{model_name} eval', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(images)
        main_logits, _ = unpack_outputs(outputs)
        preds = torch.argmax(main_logits, dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        targets_all.extend(labels.detach().cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    f1 = f1_score(targets_all, preds_all, average='weighted')
    return acc, f1

## Training Runner


In [11]:
def run_experiment(model_name: str, loss_name: str):
    print(f'\Training {model_name.upper()} with {loss_name}')

    loaders = inception_loaders if model_name == 'inception' else standard_loaders
    model = get_model(model_name, num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
    loss_fn = make_loss(loss_name)

    history = []
    best_state = None
    best_val_f1 = -1.0

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch(model, loaders['train'], optimizer, loss_fn, model_name)
        val_acc, val_f1 = evaluate(model, loaders['val'], model_name)
        scheduler.step(val_f1)

        current_lr = optimizer.param_groups[0]['lr']
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1': train_f1,
            'val_acc': val_acc,
            'val_f1': val_f1,
            'lr': current_lr,
        })

        print(
            f'Epoch {epoch}/{cfg.epochs} | '
            f'Train Loss: {train_loss:.4f} | '
            f'Train Acc: {train_acc:.4f} | '
            f'Train F1: {train_f1:.4f} | '
            f'Val Acc: {val_acc:.4f} | '
            f'Val F1: {val_f1:.4f} | '
            f'LR: {current_lr:.6f}'
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    test_acc, test_f1 = evaluate(model, loaders['test'], model_name)
    print(f'Best Val F1: {best_val_f1:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}')

    return pd.DataFrame(history), {
        'model': model_name,
        'loss': loss_name,
        'best_val_f1': best_val_f1,
        'test_acc': test_acc,
        'test_f1': test_f1,
    }

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_7209/3640224169.py:2: SyntaxWarning: invalid escape sequence '\T'
  print(f'\Training {model_name.upper()} with {loss_name}')


In [12]:
models_to_train = ['mobilenet', 'vgg16', 'vgg19', 'inception']

all_histories = []
all_results = []

for model_name in models_to_train:
    history_df, result = run_experiment(model_name, 'ce')
    history_df['model'] = model_name
    history_df['loss'] = 'ce'
    all_histories.append(history_df)
    all_results.append(result)

\Training MOBILENET with ce


mobilenet train:   0%|          | 0/879 [00:00<?, ?it/s]

mobilenet eval:   0%|          | 0/193 [00:00<?, ?it/s]

Epoch 1/4 | Train Loss: 2.3855 | Train Acc: 0.2328 | Train F1: 0.2437 | Val Acc: 0.2890 | Val F1: 0.2947 | LR: 0.000100


mobilenet train:   0%|          | 0/879 [00:00<?, ?it/s]

...

inception train:   0%|          | 0/1758 [00:00<?, ?it/s]

inception eval:   0%|          | 0/385 [00:00<?, ?it/s]

Epoch 4/4 | Train Loss: 2.8203 | Train Acc: 0.3814 | Train F1: 0.3939 | Val Acc: 0.2734 | Val F1: 0.2868 | LR: 0.000050


inception eval:   0%|          | 0/379 [00:00<?, ?it/s]

Best Val F1: 0.2993 | Test Acc: 0.2965 | Test F1: 0.2873


In [13]:
for model_name in models_to_train:
    history_df, result = run_experiment(model_name, 'distance_ce')
    history_df['model'] = model_name
    history_df['loss'] = 'distance_ce'
    all_histories.append(history_df)
    all_results.append(result)

\Training MOBILENET with distance_ce


mobilenet train:   0%|          | 0/879 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e00561de8e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e00561de8e0>    
if w.is_alive():Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

     self._shutdown_workers() 
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
       if w.is_alive():Exception ignored in: Exception ignored in: ^
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e00561de8e0><function _MultiProcessingDataLoaderIter.__del__ at 0x7e00561de8e0>^ 

^ Traceback (most recent cal

mobilenet eval:   0%|          | 0/193 [00:00<?, ?it/s]

Epoch 1/4 | Train Loss: 2.4109 | Train Acc: 0.2369 | Train F1: 0.2509 | Val Acc: 0.2362 | Val F1: 0.2421 | LR: 0.000100


...

inception train:   0%|          | 0/1758 [00:00<?, ?it/s]

inception eval:   0%|          | 0/385 [00:00<?, ?it/s]

Epoch 4/4 | Train Loss: 2.8498 | Train Acc: 0.3791 | Train F1: 0.3916 | Val Acc: 0.3114 | Val F1: 0.3238 | LR: 0.000100


inception eval:   0%|          | 0/379 [00:00<?, ?it/s]

Best Val F1: 0.3238 | Test Acc: 0.3159 | Test F1: 0.3261


## Final Comparison


In [14]:
history_table = pd.concat(all_histories, ignore_index=True)
results_table = pd.DataFrame(all_results).sort_values(['loss', 'test_f1'], ascending=[True, False]).reset_index(drop=True)

display(results_table)

,model,loss,best_val_f1,test_acc,test_f1
0,mobilenet,ce,0.311265,0.306537,0.313009
1,vgg19,ce,0.304929,0.290855,0.305084
2,inception,ce,0.299317,0.296467,0.287297
3,vgg16,ce,0.278499,0.276824,0.277670
4,inception,distance_ce,0.323812,0.315946,0.326109
5,vgg16,distance_ce,0.318052,0.309343,0.314798
6,mobilenet,distance_ce,0.303800,0.291350,0.298770
7,vgg19,distance_ce,0.276036,0.264114,0.272330


In [15]:
pivot_table = results_table.pivot(index='model', columns='loss', values=['test_acc', 'test_f1'])
display(pivot_table)

test_acc               test_f1            
loss             ce distance_ce        ce distance_ce
model                                                
inception  0.296467    0.315946  0.287297    0.326109
mobilenet  0.306537    0.291350  0.313009    0.298770
vgg16      0.276824    0.309343  0.277670    0.314798
vgg19      0.290855    0.264114  0.305084    0.272330

## 1. Methodology

This work addresses embryo stage classification as a multiclass image classification problem using pretrained convolutional neural networks. The dataset was prepared by sampling **1 frame every 8 frames** from each annotated stage interval in order to reduce redundancy while still preserving stage progression information. Since the embryo stages are sequential and ordered, class labels were treated as **ordinal categories** rather than completely unrelated classes. In preprocessing, class **15 was merged into class 14**, giving an effective total of **15 classes**. To avoid data leakage, the split was performed **embryo-wise**, so that frames from the same embryo do not appear across train, validation, and test sets.

Four pretrained CNN architectures were used:
- **MobileNetV2**
- **VGG16**
- **VGG19**
- **InceptionV3**

For each model, two training settings were evaluated:
- **Baseline:** weighted CrossEntropy loss with label smoothing
- **Custom loss:** CrossEntropy plus an ordinal distance-aware penalty

The motivation for the custom loss is that in embryo development, misclassifying a stage as a **nearby stage** should be penalized less than misclassifying it as a **far-away stage**. Standard CrossEntropy only rewards the correct class probability and does not explicitly account for the ordered nature of the labels. To incorporate this property, the following loss was used:

**L = L_CE + λ * L_dist**

where

**L_dist = 1/{(C-1)^2} * 1/N * ( \sum_{i=1}^{N} \sum_{k=0}^{C-1} p_{ik}*(k-y_i)^2)**


Here:
- \(L_{CE}\) is weighted CrossEntropy loss
- \(p_{ik}\) is the softmax probability of class \(k\) for sample \(i\)
- \(y_i\) is the true class index
- \(C\) is the number of classes
- \((k-y_i)^2\) measures how far the predicted class is from the true stage
- \(\lambda\) controls the contribution of the distance penalty

Thus, the final implemented loss is:

\[
**L = L_{CE} + 0.25 * L_{dist}**
\]

The value **\(\lambda = 0.25\)** was chosen so that CrossEntropy remains the main training objective, while the distance term acts as a moderate regularizer encouraging ordinal consistency. A much larger value could make the model focus too strongly on distance and weaken standard classification learning, while a much smaller value would reduce the benefit of modeling stage order. This custom loss is **fully differentiable**, because it is composed of differentiable operations such as softmax, multiplication, addition, averaging, and CrossEntropy. Therefore, it can be optimized directly using standard backpropagation.

---

## 2. Results

The baseline CrossEntropy and the custom distance-aware loss were evaluated on all four models using **test accuracy** and **weighted F1-score**.

| Model | CE Test Accuracy | Distance-CE Test Accuracy | CE Test F1 | Distance-CE Test F1 |
|---|---:|---:|---:|---:|
| InceptionV3 | 0.2965 | **0.3159** | 0.2873 | **0.3261** |
| MobileNetV2 | **0.3065** | 0.2914 | **0.3130** | 0.2988 |
| VGG16 | 0.2768 | **0.3093** | 0.2777 | **0.3148** |
| VGG19 | **0.2909** | 0.2641 | **0.3051** | 0.2723 |

The results show that the custom loss was **beneficial for InceptionV3 and VGG16**, but not for MobileNetV2 and VGG19. The strongest overall performance was obtained by **InceptionV3 with the custom distance-aware loss**, which achieved the best test accuracy (**0.3159**) and best test weighted F1-score (**0.3261**). This suggests that for a sufficiently expressive model, incorporating ordinal stage distance improves learning and leads to better generalization.

For **VGG16**, the custom loss also produced a clear improvement over the CE baseline, indicating that explicitly penalizing distant stage errors helped the model align better with the sequential structure of embryo development. However, for **MobileNetV2** and **VGG19**, the baseline CE loss performed better. This may indicate that the effect of the distance term depends on model capacity, optimization dynamics, or how confidently each architecture distributes probability across neighboring classes.

Overall, the experiments show that the proposed custom loss is meaningful for this task because it encodes an important desirable property: **stage-order awareness**. While it does not improve every architecture uniformly, it delivers the best overall result with InceptionV3 and demonstrates that modeling ordinal relationships can be advantageous in embryo stage classification.
